# 08 - Soumissions de contrôle

Objectif : soumettre un échantillon de modèles allant du plus simple au plus complet, pour vérifier que le classement
obtenu en CV se retrouve sur le score public (étape e).

Règle fixée **avant** de voir les scores : la soumission retenue pour le classement privé reste `final_blend`, quels que soient
les résultats publics. Choisir sur le public reviendrait à sélectionner sur ~16 000 lignes bruitées (±0,5 point).

Chaque modèle est réentraîné sur tout le train. Compter 5 à 8 minutes.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

import utils
from utils import RET, SV

X, y, Xt = utils.load_data()
up = (y > 0).to_numpy()
ALLOCS = np.array(sorted(X.ALLOCATION.unique()))
code = lambda df: pd.Series(np.arange(len(ALLOCS)), index=ALLOCS)[df.ALLOCATION].to_numpy()
dummies = lambda df: sparse.csr_matrix((np.ones(len(df)), (np.arange(len(df)), code(df))), shape=(len(df), len(ALLOCS)))

sample = pd.read_csv(utils.find_data_dir() / 'sample_submission.csv', index_col='ROW_ID')
Path('submissions').mkdir(exist_ok=True)
def save(name, score):
    sub = pd.DataFrame({sample.columns[0]: (np.asarray(score) > 0).astype(int)}, index=Xt.index).loc[sample.index]
    sub.index.name = 'ROW_ID'
    sub.to_csv(f'submissions/{name}.csv')
    print(f'{name:28s} part de 1 : {sub.iloc[:, 0].mean():.3f}')

## 1. Les modèles simples

In [2]:
# règle naïve : signe du rendement d'hier
save('05_naive_ret1', Xt.RET_1.to_numpy())

# benchmark officiel QRT
def benchmark_features(df):
    df = df.copy()
    for i in [3, 5, 10, 15, 20]:
        df[f'AVERAGE_PERF_{i}'] = df[RET[:i]].mean(axis=1)
        df[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    df['STD_PERF_20'] = df[RET].std(axis=1)
    df['ALLOCATIONS_STD_PERF_20'] = df.groupby('TS')['STD_PERF_20'].transform('mean')
    return df
FB, FBt = benchmark_features(X), benchmark_features(Xt)
cols = [c for c in FB.columns if c not in ('TS', 'ALLOCATION', 'GROUP')]
bench = lgb.train(dict(objective='mse', seed=42, verbosity=-1, learning_rate=1e-2, max_depth=3),
                  lgb.Dataset(FB[cols], label=y.to_numpy()), num_boost_round=500)
save('06_benchmark_lgbm', bench.predict(FBt[cols]))

05_naive_ret1                part de 1 : 0.515


06_benchmark_lgbm            part de 1 : 0.589


## 2. Les deux composantes du mélange final, séparément

In [3]:
Wt, Wc = utils.peer_weights(X)
L = pd.concat([utils.baseline_features(X), utils.peer_features(X, Wt, Wc), utils.cross_sectional_features(X)], axis=1)
Lt = pd.concat([utils.baseline_features(Xt), utils.peer_features(Xt, Wt, Wc), utils.cross_sectional_features(Xt)], axis=1)
s = StandardScaler().fit(L)
logit = LogisticRegression(C=0.003, max_iter=3000).fit(sparse.hstack([s.transform(L), dummies(X)]).tocsr(), up)
save('07_logit_s5', logit.predict_proba(sparse.hstack([s.transform(Lt), dummies(Xt)]).tocsr())[:, 1] - .5)

G = pd.concat([utils.rich_features(X), utils.peer_features(X, Wt, Wc), utils.cross_sectional_features(X)], axis=1)
Gt = pd.concat([utils.rich_features(Xt), utils.peer_features(Xt, Wt, Wc), utils.cross_sectional_features(Xt)], axis=1)
G['alloc'], Gt['alloc'] = code(X), code(Xt)
params = dict(objective='binary', verbosity=-1, seed=42, max_depth=3, num_leaves=8, learning_rate=.02,
              min_data_in_leaf=2000, feature_fraction=.8, bagging_fraction=.7, bagging_freq=1, lambda_l2=10, cat_smooth=50)
booster = lgb.train(params, lgb.Dataset(G, label=up.astype(int), categorical_feature=['alloc']), num_boost_round=500)
save('08_lgbm_s5', booster.predict(Gt) - .5)

07_logit_s5                  part de 1 : 0.583


08_lgbm_s5                   part de 1 : 0.575


## 3. Le réseau de neurones bridé

In [4]:
def window(df):
    vol = df[RET].std(axis=1).replace(0, np.nan)
    return pd.concat([df[RET].div(vol, axis=0).fillna(0).add_prefix('z_'),
                      df[SV].groupby(df.TS).rank(pct=True).fillna(.5).add_prefix('r_')], axis=1)
N = pd.concat([window(X), L], axis=1)
Nt = pd.concat([window(Xt), Lt], axis=1)
sn = StandardScaler().fit(N)
mlp = MLPClassifier(hidden_layer_sizes=(16,), alpha=1.0, max_iter=5, batch_size=2048, learning_rate_init=1e-3,
                    random_state=42).fit(sparse.hstack([sn.transform(N), dummies(X)]).tocsr(), up)
save('09_mlp_bride', mlp.predict_proba(sparse.hstack([sn.transform(Nt), dummies(Xt)]).tocsr())[:, 1] - .5)

09_mlp_bride                 part de 1 : 0.596


## 4. Tableau à compléter avec les scores publics

Les fichiers `02_logit_alloc.csv` et `03_blend_logit_lgbm.csv` ont été créés par les notebooks 04 et 05.

In [5]:
j = utils.load_journal().set_index('modele')
plan = pd.DataFrame([
    ('05_naive_ret1', 'naive_ret1'), ('06_benchmark_lgbm', 'benchmark_lgbm'), ('01_logit_baseline', 'logit_baseline'),
    ('02_logit_alloc', 'logit_alloc'), ('03_blend_logit_lgbm', 'blend_logit_lgbm'), ('07_logit_s5', 'final_logit_s5'),
    ('08_lgbm_s5', None), ('09_mlp_bride', 'mlp_bride'), ('04_final_blend', 'final_blend')],
    columns=['fichier', 'journal'])
plan['CV dates récentes'] = plan.journal.map(lambda k: j.loc[k, 'recent_moy'] if k in j.index else np.nan).round(4)
plan.loc[plan.fichier == '08_lgbm_s5', 'CV dates récentes'] = 0.5215   # LightGBM seul, notebook 06 section 3
plan['score public'] = np.nan
plan.loc[plan.fichier == '01_logit_baseline', 'score public'] = 0.5064
plan.loc[plan.fichier == '04_final_blend', 'score public'] = 0.5122
plan

,fichier,journal,CV dates récentes,score public
0,05_naive_ret1,naive_ret1,0.5157,NaN
1,06_benchmark_lgbm,benchmark_lgbm,0.5184,NaN
2,01_logit_baseline,logit_baseline,0.5173,0.5064
3,02_logit_alloc,logit_alloc,0.5189,NaN
4,03_blend_logit_lgbm,blend_logit_lgbm,0.5203,NaN
5,07_logit_s5,final_logit_s5,0.5221,NaN
6,08_lgbm_s5,NaN,0.5215,NaN
7,09_mlp_bride,mlp_bride,0.5138,NaN
8,04_final_blend,final_blend,0.5236,0.5122
